# Cricket Batting Shot Classifier — Full Pipeline (Stage 1 + Stage 2 + Visualization)

One notebook, same structure as the bowling one: training (Stage 1) → reference
comparison/report (Stage 2) → skeleton-overlay visualization for demos.

**Before running:** Runtime → Change runtime type → GPU.


## 1. Install dependencies

In [1]:
!pip install -q kaggle mediapipe opencv-python-headless scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 14.9 MB/s eta 0:00:00


## 2. Imports

In [2]:
import os
import cv2
import json
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision as mp_vision
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle

print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TF version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## MediaPipe pose model setup (Tasks API)

Recent MediaPipe pip builds removed the legacy `mp.solutions` API — this loads the current Tasks-API pose model instead. Run this once before any extraction/visualization cells.

In [3]:
!wget -q -O /content/pose_landmarker.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task

base_options = mp_tasks.BaseOptions(model_asset_path="/content/pose_landmarker.task")
pose_options = mp_vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=mp_vision.RunningMode.IMAGE,
    min_pose_detection_confidence=0.4,
    min_tracking_confidence=0.4,
)
landmarker = mp_vision.PoseLandmarker.create_from_options(pose_options)


## 3. Load the dataset from Kaggle

Downloads **aneesh10/cricket-shot-dataset** directly from Kaggle. You'll be prompted to
upload your `kaggle.json` API token (Kaggle -> your profile -> **Settings** -> **API** ->
**Create New Token**, which downloads `kaggle.json`).

This dataset is a flat `data/<class>/*.jpg` layout (no train/val/test split), with 4 classes:
`drive`, `legglance-flick`, `pullshot`, `sweep`. The split into train/val/test happens later,
in section 6, once `CLASS_NAMES` is defined.


In [4]:
import os

print("Upload your kaggle.json (Kaggle -> Settings -> API -> Create New Token)")
from google.colab import files
uploaded = files.upload()  # expects kaggle.json

os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download -d aneesh10/cricket-shot-dataset -p /content/cricket-shot-raw --unzip

RAW_DATASET_ROOT = "/content/cricket-shot-raw/data"
print("Raw dataset root:", RAW_DATASET_ROOT)
print("Classes found:", sorted(os.listdir(RAW_DATASET_ROOT)))

# Snapshot what exists in /content right now (raw dataset + kaggle.json included) so the
# final cell can later zip up ONLY what this runtime generates from here on — this also
# means kaggle.json (your credential) never ends up in that downloadable zip.
_RUNTIME_BASELINE = set(os.listdir("/content"))


Upload your kaggle.json (Kaggle -> Settings -> API -> Create New Token)


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/aneesh10/cricket-shot-dataset
License(s): unknown
100% 645M/645M [00:06<00:00, 103MB/s] 

Raw dataset root: /content/cricket-shot-raw/data
Classes found: ['drive', 'legglance-flick', 'pullshot', 'sweep']


## 4. Config

In [5]:
CLASS_NAMES = ["drive", "legglance-flick", "pullshot", "sweep"]  # the 4 Kaggle class folders

SEQUENCE_LENGTH = 30      # frames per input sequence (see extract_keypoint_sequence_from_image below)
NUM_KEYPOINTS = 33        # MediaPipe Pose gives 33 landmarks; we'll select the 17 we care about below
FEATURES_PER_KEYPOINT = 3 # x, y, visibility

# The 17 joint indices we're keeping (matches the 17-joint setup used in the Asana Sense project).
# MediaPipe Pose landmark indices — see https://developers.google.com/mediapipe/solutions/vision/pose_landmarker
KEEP_LANDMARKS = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 0]
# left/right shoulder, elbow, wrist, hip, knee, ankle, heel, foot_index (both sides) + nose

# Named lookup into KEEP_LANDMARKS/each feature row, in the same order as the list above.
# (Needed by the Stage 2 metric functions in section 6 further down — was referenced there
# but never defined in the original notebook, so this fixes that.)
JOINT_IDX = {
    "l_shoulder": 0, "r_shoulder": 1, "l_elbow": 2, "r_elbow": 3,
    "l_wrist": 4, "r_wrist": 5, "l_hip": 6, "r_hip": 7,
    "l_knee": 8, "r_knee": 9, "l_ankle": 10, "r_ankle": 11,
    "l_heel": 12, "r_heel": 13, "l_foot_index": 14, "r_foot_index": 15, "nose": 16,
}

NUM_FEATURES = len(KEEP_LANDMARKS) * FEATURES_PER_KEYPOINT
print("Feature vector length per frame:", NUM_FEATURES)


Feature vector length per frame: 51


## 5. Pose extraction: image/video -> keypoint sequence

`extract_keypoint_sequence` (video, unchanged) samples `SEQUENCE_LENGTH` frames across a clip.

`extract_keypoint_sequence_from_image` (new) runs pose detection once on a still image and repeats
that single frame's keypoints `SEQUENCE_LENGTH` times, so a static image produces the same input
shape `(SEQUENCE_LENGTH, NUM_FEATURES)` the GRU model expects — no architecture changes needed.
Since there's no real motion in a single photo, this is a deliberate stand-in, not real temporal
information; it lets the Kaggle image dataset train the existing model as-is.


In [6]:
def extract_keypoint_sequence(video_path, sequence_length=SEQUENCE_LENGTH):
    """cuDNN's fused GRU kernel only accepts masks that are right-padded — real frames
    first, then zero-padding only at the very end, never a zero row in the middle. A
    single frame with no detected pose (motion blur, brief occlusion, bad angle, etc.)
    used to get filled with zeros wherever it happened in the sequence, which breaks
    that pattern and crashes inference. Fix: forward-fill (and backward-fill the very
    first frame, if needed) from the nearest successfully-detected frame instead, so
    every "real" frame in the clip carries real keypoints. True padding only happens
    at the end, if the clip has fewer than sequence_length readable frames."""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return None
    frame_indices = np.linspace(0, max(total_frames - 1, 0), sequence_length).astype(int)
    sequence = []       # list of (feats or None)
    frame_idx = 0
    wanted = list(frame_indices)
    next_i = 0
    while cap.isOpened() and next_i < len(wanted):
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx == wanted[next_i]:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = landmarker.detect(mp_image)
            if result.pose_landmarks:
                lm = result.pose_landmarks[0]
                frame_feats = []
                for idx in KEEP_LANDMARKS:
                    p = lm[idx]
                    frame_feats.extend([p.x, p.y, p.visibility])
            else:
                frame_feats = None  # no pose detected on this sampled frame
            sequence.append(frame_feats)
            next_i += 1
        frame_idx += 1
    cap.release()
    if len(sequence) == 0 or all(f is None for f in sequence):
        return None

    # Forward-fill missing frames from the last successful detection...
    last_good = None
    for i in range(len(sequence)):
        if sequence[i] is not None:
            last_good = sequence[i]
        elif last_good is not None:
            sequence[i] = last_good
    # ...then backward-fill any still-missing frames at the very start of the clip.
    first_good = next(f for f in sequence if f is not None)
    for i in range(len(sequence)):
        if sequence[i] is None:
            sequence[i] = first_good

    seq = np.array(sequence, dtype=np.float32)
    if seq.shape[0] < sequence_length:
        # Genuine end-of-clip padding (short video) — zeros here are safe for cuDNN
        # since they're contiguous at the end, not scattered through the middle.
        pad = np.zeros((sequence_length - seq.shape[0], NUM_FEATURES), dtype=np.float32)
        seq = np.vstack([seq, pad])
    return seq

def extract_keypoint_sequence_from_image(image_path, sequence_length=SEQUENCE_LENGTH):
    """Images have no motion to sample across, so we run pose detection once and repeat
    that single frame's keypoints for the full sequence length — keeps the input shape
    identical to the video pipeline without touching the model."""
    frame = cv2.imread(image_path)
    if frame is None:
        return None
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect(mp_image)
    if not result.pose_landmarks:
        return None
    lm = result.pose_landmarks[0]
    frame_feats = []
    for idx in KEEP_LANDMARKS:
        p = lm[idx]
        frame_feats.extend([p.x, p.y, p.visibility])
    frame_feats = np.array(frame_feats, dtype=np.float32)
    seq = np.tile(frame_feats, (sequence_length, 1))  # repeat the same frame sequence_length times
    return seq


## 6. Build the dataset (split + features + labels)

The Kaggle dataset has no train/val/test folders — just one folder per class — so this cell
first does a stratified 70/15/15 split per class (in memory, over file paths), then extracts
pose keypoints for every image and builds `X_train, y_train`, `X_val, y_val`, `X_test, y_test`.

**This is the slow step** — pose extraction on ~4,700 images takes a while even on GPU
(MediaPipe pose itself runs on CPU per-frame). The next cell saves the resulting `.npy` arrays
so you don't have to redo it if the runtime disconnects.


In [7]:
from sklearn.model_selection import train_test_split

IMAGE_EXTS = (".jpg", ".jpeg", ".png")
SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}  # stratified per class

split_files = {"train": {}, "val": {}, "test": {}}
for class_name in CLASS_NAMES:
    class_dir = os.path.join(RAW_DATASET_ROOT, class_name)
    if not os.path.isdir(class_dir):
        print(f"WARNING: missing folder {class_dir}, skipping class")
        continue
    all_files = [os.path.join(class_dir, f) for f in os.listdir(class_dir)
                 if f.lower().endswith(IMAGE_EXTS)]
    train_f, temp_f = train_test_split(all_files, train_size=SPLIT_RATIOS["train"], random_state=42)
    val_frac = SPLIT_RATIOS["val"] / (SPLIT_RATIOS["val"] + SPLIT_RATIOS["test"])
    val_f, test_f = train_test_split(temp_f, train_size=val_frac, random_state=42)
    split_files["train"][class_name] = train_f
    split_files["val"][class_name] = val_f
    split_files["test"][class_name] = test_f
    print(f"{class_name}: {len(train_f)} train / {len(val_f)} val / {len(test_f)} test")


def build_split(split_name):
    X, y = [], []
    skipped = 0
    for class_name, paths in split_files[split_name].items():
        for path in paths:
            seq = extract_keypoint_sequence_from_image(path)
            if seq is None:
                skipped += 1
                continue
            X.append(seq)
            y.append(class_name)
    print(f"{split_name}: {len(X)} images loaded, {skipped} skipped (no pose detected)")
    return np.array(X, dtype=np.float32), np.array(y)

X_train_raw, y_train_raw = build_split("train")
X_val_raw, y_val_raw = build_split("val")
X_test_raw, y_test_raw = build_split("test")


drive: 856 train / 183 val / 184 test
legglance-flick: 784 train / 168 val / 168 test
pullshot: 882 train / 189 val / 189 test
sweep: 784 train / 168 val / 168 test
train: 2510 images loaded, 796 skipped (no pose detected)
val: 534 images loaded, 174 skipped (no pose detected)
test: 549 images loaded, 160 skipped (no pose detected)


In [8]:
# Save extracted features so you don't have to redo pose extraction if the runtime resets
np.save("/content/X_train.npy", X_train_raw)
np.save("/content/y_train.npy", y_train_raw)
np.save("/content/X_val.npy", X_val_raw)
np.save("/content/y_val.npy", y_val_raw)
np.save("/content/X_test.npy", X_test_raw)
np.save("/content/y_test.npy", y_test_raw)
print("Saved extracted keypoint sequences to /content/*.npy")

# To reload later without re-running extraction:
# X_train_raw = np.load("/content/X_train.npy")
# y_train_raw = np.load("/content/y_train.npy", allow_pickle=True)
# ... etc for val/test


Saved extracted keypoint sequences to /content/*.npy


## 7. Encode labels

In [9]:
label_encoder = LabelEncoder()
label_encoder.fit(CLASS_NAMES)

y_train = label_encoder.transform(y_train_raw)
y_val = label_encoder.transform(y_val_raw)
y_test = label_encoder.transform(y_test_raw)

X_train, y_train = shuffle(X_train_raw, y_train, random_state=42)
X_val, y_val = X_val_raw, y_val
X_test, y_test = X_test_raw, y_test

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)
print("Classes:", list(label_encoder.classes_))


Train: (2510, 30, 51) (2510,)
Val:   (534, 30, 51) (534,)
Test:  (549, 30, 51) (549,)
Classes: [np.str_('drive'), np.str_('legglance-flick'), np.str_('pullshot'), np.str_('sweep')]


## 8. Model — GRU over keypoint sequences

Small, fast to train relative to a CNN-on-raw-frames approach — appropriate for the
time budget. `Masking` lets the model ignore zero-padded frames from short clips.


In [10]:
num_classes = len(CLASS_NAMES)

model = keras.Sequential([
    layers.Input(shape=(SEQUENCE_LENGTH, NUM_FEATURES)),
    layers.Masking(mask_value=0.0),
    layers.GRU(128, return_sequences=True),
    layers.Dropout(0.3),
    layers.GRU(64),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dense(num_classes, activation="softmax"),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking (Masking)               │ (None, 30, 51)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 30, 128)        │        69,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 111,172 (434.27 KB)

 Trainable params: 111,172 (434.27 KB)

 Non-trainable params: 0 (0.00 B)

## 9. Train

In [11]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("/content/best_batting_model.keras",
                                     monitor="val_accuracy", save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=32,
    callbacks=callbacks,
)


Epoch 1/60
79/79 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.3159 - loss: 1.3610 - val_accuracy: 0.4026 - val_loss: 1.1914 - learning_rate: 0.0010
Epoch 2/60
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4319 - loss: 1.2121 - val_accuracy: 0.4850 - val_loss: 1.1121 - learning_rate: 0.0010
Epoch 3/60
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4876 - loss: 1.1190 - val_accuracy: 0.5019 - val_loss: 1.0741 - learning_rate: 0.0010
Epoch 4/60
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5223 - loss: 1.0750 - val_accuracy: 0.6067 - val_loss: 0.9603 - learning_rate: 0.0010
Epoch 5/60
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5363 - loss: 1.0386 - val_accuracy: 0.5936 - val_loss: 0.9481 - learning_rate: 0.0010
Epoch 6/60
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5637 - loss: 1.0085 - val_accuracy: 0.5318 - val_loss: 1.0386 - learning_rate: 0.0010
Epoch 7/60
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5781 - loss: 0.9849 - val_acc

## 10. Evaluate on the held-out test set

In [12]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")

from sklearn.metrics import classification_report, confusion_matrix

y_pred = np.argmax(model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))


18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8288 - loss: 0.4608
Test accuracy: 0.8288
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
                 precision    recall  f1-score   support

          drive       0.88      0.83      0.85       135
legglance-flick       0.77      0.77      0.77       140
       pullshot       0.83      0.83      0.83       146
          sweep       0.84      0.89      0.86       128

       accuracy                           0.83       549
      macro avg       0.83      0.83      0.83       549
   weighted avg       0.83      0.83      0.83       549

Confusion matrix:
[[112   8   6   9]
 [  6 108  16  10]
 [  7  15 121   3]
 [  2   9   3 114]]


## 11. Save the final model + label encoder

Download both — the backend (FastAPI) will need the `.keras` file to load the model,
and the class name list to map predictions back to shot names.


In [13]:
model.save("/content/batting_shot_classifier.keras")

with open("/content/batting_class_names.json", "w") as f:
    json.dump(list(label_encoder.classes_), f)

from google.colab import files
files.download("/content/batting_shot_classifier.keras")
files.download("/content/batting_class_names.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
# Stage 2 — Reference comparison + report generation

Continues directly from Stage 1 above — `model`, `CLASS_NAMES` (as `label_encoder.classes_`),
and the keypoint-extraction function are already in memory, no need to reload anything.


In [14]:
# Stage 1 already defined CLASS_NAMES and the trained `model` above — reused directly here.
print("Using classes:", CLASS_NAMES)


Using classes: ['drive', 'legglance-flick', 'pullshot', 'sweep']


In [15]:
!pip install -q fastdtw scipy
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 5. Build the reference library ("good form" images per shot)

Auto-picks one image per class from the training split (`split_files["train"]`, built in section 6)
as the "good form" reference — edit `REFERENCE_CLIPS` below if you'd rather hand-pick specific
reference images per class.

**Caveat:** since these references (and the training data) are single images repeated across the
sequence, DTW/motion-based comparisons in Stage 2 are only meaningful if what you compare them
against is *also* a static-pose-style clip. Comparing against a real moving video (like the sample
video test in section 8 below) will look distorted, since there's no real motion in the reference.

**This only needs to be run once** (or whenever you change the reference images) — the resulting
sequences get cached to disk.


In [16]:
# Auto-picked: one reference image per class from the training split (built in section 6).
# Edit any entry to point at a different image path if you want to hand-pick a cleaner example.
REFERENCE_CLIPS = {class_name: paths[0] for class_name, paths in split_files["train"].items() if paths}

REFERENCE_CACHE_PATH = "/content/reference_library.npz"

def build_reference_library():
    ref_seqs = {}
    for shot, path in REFERENCE_CLIPS.items():
        if not os.path.exists(path):
            print(f"WARNING: reference image missing for '{shot}' at {path} — skipping")
            continue
        seq = extract_keypoint_sequence_from_image(path)
        if seq is not None:
            ref_seqs[shot] = seq
        else:
            print(f"WARNING: could not extract keypoints for reference '{shot}'")
    return ref_seqs

if os.path.exists(REFERENCE_CACHE_PATH):
    cached = np.load(REFERENCE_CACHE_PATH)
    REFERENCE_LIBRARY = {k: cached[k] for k in cached.files}
    print("Loaded cached reference library:", list(REFERENCE_LIBRARY.keys()))
else:
    REFERENCE_LIBRARY = build_reference_library()
    np.savez(REFERENCE_CACHE_PATH, **REFERENCE_LIBRARY)
    print("Built and cached reference library:", list(REFERENCE_LIBRARY.keys()))


Built and cached reference library: ['drive', 'legglance-flick', 'pullshot']


## 6. Metric functions (Stage 2 core logic)

Everything here compares the user's sequence to the matching reference sequence.
Nothing here claims an absolute clinical score — every number is a *deviation from
the reference*, framed and labeled accordingly (see the honesty notes at the end).


In [17]:
def get_joint_xy(seq, frame_idx, joint_name):
    base = JOINT_IDX[joint_name] * FEATURES_PER_KEYPOINT
    x, y = seq[frame_idx, base], seq[frame_idx, base + 1]
    return np.array([x, y])

def angle_at(seq, frame_idx, a, b, c):
    """Angle at joint b, formed by points a-b-c, in degrees."""
    pa, pb, pc = get_joint_xy(seq, frame_idx, a), get_joint_xy(seq, frame_idx, b), get_joint_xy(seq, frame_idx, c)
    v1, v2 = pa - pb, pc - pb
    denom = (np.linalg.norm(v1) * np.linalg.norm(v2)) + 1e-8
    cos_angle = np.clip(np.dot(v1, v2) / denom, -1.0, 1.0)
    return np.degrees(np.arccos(cos_angle))

def dtw_align(user_seq, ref_seq):
    """Aligns the two sequences frame-by-frame using DTW on flattened keypoints.
    Returns (distance, path) where path is a list of (user_idx, ref_idx) pairs."""
    distance, path = fastdtw(user_seq, ref_seq, dist=euclidean)
    return distance, path

def movement_quality_score(user_seq, ref_seq, dtw_distance):
    """Lower DTW distance = closer match to reference. Converted to a 0-100
    similarity number for readability. This is a RELATIVE similarity measure,
    not a calibrated fitness score."""
    # Normalize by sequence length and a rough scale constant; tune SCALE after
    # seeing real distance values on your actual data.
    SCALE = 50.0
    normalized = dtw_distance / (len(user_seq) * SCALE)
    similarity = max(0.0, 100.0 - normalized * 100.0)
    return round(similarity, 1)

def symmetry_score(seq, frame_idx):
    """Compares left vs right elbow and knee angles at a given frame.
    Smaller difference = more symmetric."""
    l_elbow = angle_at(seq, frame_idx, "l_shoulder", "l_elbow", "l_wrist")
    r_elbow = angle_at(seq, frame_idx, "r_shoulder", "r_elbow", "r_wrist")
    l_knee = angle_at(seq, frame_idx, "l_hip", "l_knee", "l_ankle")
    r_knee = angle_at(seq, frame_idx, "r_hip", "r_knee", "r_ankle")
    elbow_diff = abs(l_elbow - r_elbow)
    knee_diff = abs(l_knee - r_knee)
    avg_diff = (elbow_diff + knee_diff) / 2.0
    score = max(0.0, 100.0 - avg_diff)  # rough scale; tune after seeing real values
    return round(score, 1), {"elbow_angle_diff_deg": round(elbow_diff, 1),
                              "knee_angle_diff_deg": round(knee_diff, 1)}

def coordination_deviation(user_seq, ref_seq):
    """Compares the TIMING of hip rotation vs shoulder rotation (a simple proxy
    for kinetic-chain sequencing) between user and reference."""
    def rotation_signal(seq, joint_a, joint_b):
        # crude proxy: x-distance between left/right joint pair over time,
        # as a stand-in for rotation amount frame-by-frame
        return np.array([abs(get_joint_xy(seq, i, joint_a)[0] - get_joint_xy(seq, i, joint_b)[0])
                          for i in range(len(seq))])

    user_hip = rotation_signal(user_seq, "l_hip", "r_hip")
    user_shoulder = rotation_signal(user_seq, "l_shoulder", "r_shoulder")
    ref_hip = rotation_signal(ref_seq, "l_hip", "r_hip")
    ref_shoulder = rotation_signal(ref_seq, "l_shoulder", "r_shoulder")

    user_lag = np.argmax(user_shoulder) - np.argmax(user_hip)
    ref_lag = np.argmax(ref_shoulder) - np.argmax(ref_hip)
    lag_deviation_frames = abs(user_lag - ref_lag)
    return lag_deviation_frames

def mobility_proxy(user_seq, ref_seq, path):
    """APPROXIMATE. Compares front-knee flexion at the DTW-aligned peak-flexion
    frame. Camera-angle dependent — see caveats."""
    user_knee_angles = [angle_at(user_seq, i, "l_hip", "l_knee", "l_ankle") for i in range(len(user_seq))]
    ref_knee_angles = [angle_at(ref_seq, i, "l_hip", "l_knee", "l_ankle") for i in range(len(ref_seq))]
    user_min_idx = int(np.argmin(user_knee_angles))
    # find the aligned reference frame for that user frame via the DTW path
    aligned_ref_idx = next((r for (u, r) in path if u == user_min_idx), int(np.argmin(ref_knee_angles)))
    deviation_deg = abs(user_knee_angles[user_min_idx] - ref_knee_angles[aligned_ref_idx])
    return round(deviation_deg, 1)

def balance_proxy(user_seq, ref_seq):
    """WEAK PROXY ONLY. Sideways drift of the hip midpoint over the clip,
    compared to the reference's drift. Not a real balance/COM measurement."""
    def hip_midpoint_drift(seq):
        xs = [(get_joint_xy(seq, i, "l_hip")[0] + get_joint_xy(seq, i, "r_hip")[0]) / 2 for i in range(len(seq))]
        return np.std(xs)
    user_drift = hip_midpoint_drift(user_seq)
    ref_drift = hip_midpoint_drift(ref_seq)
    return round(float(user_drift - ref_drift), 4)

def core_stability_proxy(user_seq, ref_seq):
    """WEAK PROXY ONLY. Trunk-angle (hip-mid to shoulder-mid vector) deviation
    from vertical, variance across the clip vs reference. Not a real core
    strength/stability measurement."""
    def trunk_angle_variance(seq):
        angles = []
        for i in range(len(seq)):
            hip_mid = (get_joint_xy(seq, i, "l_hip") + get_joint_xy(seq, i, "r_hip")) / 2
            shoulder_mid = (get_joint_xy(seq, i, "l_shoulder") + get_joint_xy(seq, i, "r_shoulder")) / 2
            vec = shoulder_mid - hip_mid
            angle = np.degrees(np.arctan2(vec[0], -vec[1]))  # deviation from vertical
            angles.append(angle)
        return np.var(angles)
    user_var = trunk_angle_variance(user_seq)
    ref_var = trunk_angle_variance(ref_seq)
    return round(float(user_var - ref_var), 2)


## 7. The integrated pipeline function

**This is the one function your backend teammates call.** Give them this cell's code —
`generate_report(video_path)` handles Stage 1 + Stage 2 end to end and returns one JSON-ready dict.


In [18]:
def _json_safe(obj):
    """Recursively converts numpy scalar/array types to plain Python types so the
    report dict can be safely passed to json.dumps(). Several of the Stage 2 metric
    functions above do angle/np.std/np.var math and can leave numpy.float32 (etc.)
    values in the result — json.dumps() doesn't know how to serialize those."""
    if isinstance(obj, dict):
        return {k: _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, np.generic):        # numpy scalar (float32, int64, etc.)
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


def generate_report(video_path):
    user_seq = extract_keypoint_sequence(video_path)
    if user_seq is None:
        return {"error": "Could not extract pose from video. Check the file/camera visibility."}

    # ---- Stage 1: classify the shot ----
    pred = model.predict(np.expand_dims(user_seq, axis=0), verbose=0)[0]
    pred_idx = int(np.argmax(pred))
    shot_label = CLASS_NAMES[pred_idx]
    confidence = float(pred[pred_idx])

    report = {
        "shot_classification": {
            "label": shot_label,
            "confidence": round(confidence, 3)
        }
    }

    # ---- Stage 2: compare against reference, if we have one for this shot ----
    ref_seq = REFERENCE_LIBRARY.get(shot_label)
    if ref_seq is None:
        report["comparison"] = {"note": f"No reference clip available for '{shot_label}' yet."}
        return report

    dtw_distance, path = dtw_align(user_seq, ref_seq)
    symmetry, symmetry_detail = symmetry_score(user_seq, frame_idx=len(user_seq) // 2)

    report["comparison"] = {
        "movement_quality_similarity_pct": movement_quality_score(user_seq, ref_seq, dtw_distance),
        "symmetry_score": symmetry,
        "symmetry_detail": symmetry_detail,
        "coordination_timing_deviation_frames": int(coordination_deviation(user_seq, ref_seq)),
        "mobility_indicator_deg_deviation": mobility_proxy(user_seq, ref_seq, path),
        "balance_indicator": balance_proxy(user_seq, ref_seq),
        "core_stability_indicator": core_stability_proxy(user_seq, ref_seq),
    }

    report["caveats"] = [
        "All metrics are RELATIVE to a single reference clip, not absolute/clinical measurements.",
        "mobility_indicator, balance_indicator, and core_stability_indicator are approximate proxies, not validated biomechanical measurements.",
        "Accuracy depends on the uploaded video's camera angle roughly matching the reference clip's angle.",
    ]

    return _json_safe(report)


## 8. Try it on a sample video

In [27]:
print("Upload a test video to run through the full pipeline")
uploaded = files.upload()
test_video_path = list(uploaded.keys())[0]

report = generate_report(test_video_path)
print(json.dumps(report, indent=2))


Upload a test video to run through the full pipeline


Saving sample5.mp4 to sample5.mp4
{
  "shot_classification": {
    "label": "pullshot",
    "confidence": 0.821
  },
  "comparison": {
    "movement_quality_similarity_pct": 94.2,
    "symmetry_score": 98.80000305175781,
    "symmetry_detail": {
      "elbow_angle_diff_deg": 0.8999999761581421,
      "knee_angle_diff_deg": 1.5
    },
    "coordination_timing_deviation_frames": 0,
    "mobility_indicator_deg_deviation": 28.399999618530273,
    "balance_indicator": 0.0338,
    "core_stability_indicator": 10.8
  },
  "caveats": [
    "All metrics are RELATIVE to a single reference clip, not absolute/clinical measurements.",
    "mobility_indicator, balance_indicator, and core_stability_indicator are approximate proxies, not validated biomechanical measurements.",
    "Accuracy depends on the uploaded video's camera angle roughly matching the reference clip's angle."
  ]
}


---
## For your backend teammates (FastAPI integration)

The whole notebook boils down to one call: `generate_report(video_path)` → returns a dict.
In FastAPI, that's roughly:

```python
from fastapi import FastAPI, UploadFile
import shutil

app = FastAPI()

# Load model + reference library ONCE at startup — not per-request
model = keras.models.load_model("batting_shot_classifier.keras")
REFERENCE_LIBRARY = ...  # load the cached .npz the same way as above

@app.post("/analyze-batting")
async def analyze_batting(video: UploadFile):
    temp_path = f"/tmp/{video.filename}"
    with open(temp_path, "wb") as f:
        shutil.copyfileobj(video.file, f)
    report = generate_report(temp_path)
    return report
```

The frontend just needs to POST the uploaded video to this endpoint and render whatever
JSON comes back — the `report` dict above is already structured for that (label, confidence,
comparison metrics, caveats).

### Notes / things to actually do before this is real
- `REFERENCE_CLIPS` in section 5 is now auto-picked from the training split — swap in
  hand-picked images per class if the auto-picked ones aren't great examples.
- The scaling constants in `movement_quality_score` and `symmetry_score` (`SCALE = 50.0`,
  the `100.0 - avg_diff` line) are rough starting points — once you run this on real data,
  look at what raw values actually come out and adjust so the 0-100 range is meaningful,
  rather than everything clustering near 0 or 100.
- `coordination_deviation`'s rotation proxy is a simplification (x-distance between paired
  joints as a stand-in for rotation) — good enough for a hackathon demo, but say so plainly
  if asked how it works in judging.
- This whole notebook is batting-only, per your ask — the bowling version needs the same
  structure once your teammates have their bowling classifier and reference clips ready.


## Visual proof: draw the skeleton on the video

Overlays MediaPipe's detected pose skeleton onto the video and plays it back — purely for
demoing to judges that the pipeline is really tracking the player's body. Doesn't feed into
any of the score calculations above.


In [28]:
from IPython.display import Video, display
POSE_CONNECTIONS = [(11,12),(11,13),(13,15),(12,14),(14,16),(11,23),(12,24),
                     (23,24),(23,25),(25,27),(27,29),(29,31),(24,26),(26,28),
                     (28,30),(30,32),(15,17),(15,19),(15,21),(16,18),(16,20),
                     (16,22),(27,31),(28,32)]

def draw_pose_landmarks(frame, landmarks, width, height):
    pts = [(int(p.x * width), int(p.y * height)) for p in landmarks]
    for a, b in POSE_CONNECTIONS:
        cv2.line(frame, pts[a], pts[b], (0, 255, 150), 2)
    for x, y in pts:
        cv2.circle(frame, (x, y), 3, (0, 200, 255), -1)


def generate_annotated_report_video(video_path, output_path="/content/annotated_report.mp4",
                                     panel_side="right", panel_width=380):
    """Runs the full pipeline AND produces one video: skeleton overlay + a side
    report panel (shot name, confidence, metrics) — not scattered text on the footage."""
    report = generate_report(video_path)
    if "error" in report:
        print(report["error"])
        return None, report

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    canvas_width = width + panel_width
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (canvas_width, height))

    label_key = "shot_classification" if "shot_classification" in report else "action_classification"
    cls = report[label_key]
    comp = report.get("comparison", {})

    lines = [
        (cls["label"].upper().replace("_", " "), (255, 255, 255), 0.75, 2),
        (f"Confidence: {cls['confidence']*100:.1f}%", (180, 180, 180), 0.5, 1),
        ("", None, 0, 0),
    ]
    if comp and "movement_quality_similarity_pct" in comp:
        lines += [
            (f"Movement Quality  {comp['movement_quality_similarity_pct']}%", (120, 255, 150), 0.5, 1),
            (f"Symmetry          {comp['symmetry_score']}",                    (120, 255, 150), 0.5, 1),
            (f"Coordination dev  {comp['coordination_timing_deviation_frames']}f", (120, 210, 255), 0.5, 1),
            (f"Mobility dev      {comp['mobility_indicator_deg_deviation']}deg",   (120, 210, 255), 0.5, 1),
            (f"Balance idx       {comp['balance_indicator']}",                 (120, 180, 255), 0.5, 1),
            (f"Core idx          {comp['core_stability_indicator']}",          (120, 180, 255), 0.5, 1),
            ("", None, 0, 0),
            ("(approximate, vs reference clip)", (120, 120, 120), 0.4, 1),
        ]
    else:
        lines.append(("No reference clip set for this class yet", (120, 120, 120), 0.4, 1))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect(mp_image)
        if result.pose_landmarks:
            draw_pose_landmarks(frame, result.pose_landmarks[0], width, height)

        canvas = np.zeros((height, canvas_width, 3), dtype=np.uint8)
        if panel_side == "right":
            canvas[:, :width] = frame
            panel_x0 = width
        else:
            canvas[:, panel_width:] = frame
            panel_x0 = 0

        overlay = canvas.copy()
        cv2.rectangle(overlay, (panel_x0, 0), (panel_x0 + panel_width, height), (25, 25, 25), -1)
        canvas = cv2.addWeighted(overlay, 0.88, canvas, 0.12, 0)
        cv2.line(canvas, (panel_x0, 0), (panel_x0, height), (80, 80, 80), 2)

        y = 45
        for text, color, scale, thickness in lines:
            if text == "":
                y += 18
                continue
            cv2.putText(canvas, text, (panel_x0 + 20, y),
                        cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness, cv2.LINE_AA)
            y += int(32 * scale) + 14

        out.write(canvas)

    cap.release()
    out.release()

    h264_path = output_path.replace(".mp4", "_h264.mp4")
    os.system(f"ffmpeg -y -loglevel error -i {output_path} -vcodec libx264 {h264_path}")
    print(f"Annotated report video saved: {h264_path}")
    return h264_path, report


annotated_path, report = generate_annotated_report_video(test_video_path, panel_side="right")
print(json.dumps(report, indent=2))
display(Video(annotated_path, embed=True, width=760))


Annotated report video saved: /content/annotated_report_h264.mp4
{
  "shot_classification": {
    "label": "pullshot",
    "confidence": 0.821
  },
  "comparison": {
    "movement_quality_similarity_pct": 94.2,
    "symmetry_score": 98.80000305175781,
    "symmetry_detail": {
      "elbow_angle_diff_deg": 0.8999999761581421,
      "knee_angle_diff_deg": 1.5
    },
    "coordination_timing_deviation_frames": 0,
    "mobility_indicator_deg_deviation": 28.399999618530273,
    "balance_indicator": 0.0338,
    "core_stability_indicator": 10.8
  },
  "caveats": [
    "All metrics are RELATIVE to a single reference clip, not absolute/clinical measurements.",
    "mobility_indicator, balance_indicator, and core_stability_indicator are approximate proxies, not validated biomechanical measurements.",
    "Accuracy depends on the uploaded video's camera angle roughly matching the reference clip's angle."
  ]
}


---
## For your backend teammates (FastAPI integration)

```python
from fastapi import FastAPI, UploadFile
import shutil

app = FastAPI()

model = keras.models.load_model("batting_shot_classifier.keras")
REFERENCE_LIBRARY = ...  # load the cached reference_library.npz the same way as in this notebook

@app.post("/analyze-batting")
async def analyze_batting(video: UploadFile):
    temp_path = f"/tmp/{video.filename}"
    with open(temp_path, "wb") as f:
        shutil.copyfileobj(video.file, f)
    return generate_report(temp_path)
```

### Things to actually do before this is real
- Section 5 (`REFERENCE_CLIPS`) is now auto-picked from the training split — swap in
  hand-picked images per class if the auto-picked ones aren't great examples.
- Tune the scoring constants in `movement_quality_score` / `symmetry_score` after seeing
  real output values from your actual data.
- Everything here is batting-only — bowling has its own separate notebook
  (`bowling_full_pipeline.ipynb`), same structure.


---
## 12. Download everything generated this runtime session (zip)

Zips up every file/folder created in `/content` since the Kaggle dataset finished downloading
(section 3) — the `.npy` feature arrays, both `.keras` model files, `batting_class_names.json`,
`reference_library.npz`, any test video you uploaded plus the annotated output video, etc.
The raw Kaggle dataset itself and your `kaggle.json` credential are excluded on purpose.


In [29]:
import shutil

EXCLUDE_ALWAYS = {"kaggle.json"}  # never bundle credentials, even if re-created later

_baseline = globals().get("_RUNTIME_BASELINE", set())
current_entries = set(os.listdir("/content"))
new_entries = sorted((current_entries - _baseline) - EXCLUDE_ALWAYS)

print("Files/folders generated this runtime session:")
for entry in new_entries:
    print(" -", entry)

if not new_entries:
    print("\nNothing new found — did you run the earlier cells in this same runtime session?")
else:
    staging_dir = "/content/_runtime_outputs_staging"
    if os.path.isdir(staging_dir):
        shutil.rmtree(staging_dir)
    os.makedirs(staging_dir)

    for entry in new_entries:
        src_path = os.path.join("/content", entry)
        dst_path = os.path.join(staging_dir, entry)
        if os.path.isdir(src_path):
            shutil.copytree(src_path, dst_path)
        else:
            shutil.copy2(src_path, dst_path)

    zip_path = shutil.make_archive("/content/colab_runtime_outputs", "zip", staging_dir)
    print("\nZipped runtime outputs to:", zip_path)

    from google.colab import files
    files.download(zip_path)


Files/folders generated this runtime session:
 - X_test.npy
 - X_train.npy
 - X_val.npy
 - annotated_report.mp4
 - annotated_report_h264.mp4
 - batting_class_names.json
 - batting_shot_classifier.keras
 - best_batting_model.keras
 - reference_library.npz
 - sample1.mp4
 - sample2.mp4
 - sample3.mp4
 - sample4.mp4
 - sample5.mp4
 - y_test.npy
 - y_train.npy
 - y_val.npy

Zipped runtime outputs to: /content/colab_runtime_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>